# CRISPR sgRNA gate (iSBH)

Manual test harness for `engine.gates.crispr.CrisprGate`. **Planned, not
implemented** and **eukaryotic only** (`supported_hosts = {YEAST, HUMAN}`).
`is_compatible()` returns a `Compatibility.no(...)`; the rest print `pending Step 5`.

## The mechanism

An sgRNA's spacer is sequestered inside a hairpin, so Cas cannot be guided
anywhere. A trigger opens that hairpin (strand displacement, as in a toehold), and
the freed sgRNA directs Cas to a promoter — the map's internally-blocked sgRNA
hairpin, iSBH.

Unlike the toehold and antisense families, CRISPR acts on **transcription** and can
regulate genes already in the genome. The spacer is chosen from the **target gene**,
not from the trigger — the one family here whose sequence does not derive from the
trigger.

## Setup

In [ ]:
# Put the shared _fixtures.py on the path. It lives in the notebooks/ root, one
# level up from this gate's folder; search upward so the notebook works wherever
# Jupyter is launched. fx.bootstrap() then adds <repo>/src so `import engine...`
# resolves. No Django, no worker, no pipeline.
import sys, pathlib

for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / '_fixtures.py').exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break

import _fixtures as fx
fx.bootstrap()

## Build the gate

In [ ]:
host = fx.Host.HUMAN         # YEAST | HUMAN only — ECOLI is rejected below
gate = fx.crispr(host=host)
fx.describe_gate(gate)       # available = False

## Inputs

A `TriggerSet` (the circuit's inputs) and a `Constraints` (the researcher's
limits). Both are made up here — override any field via keyword.

In [ ]:
triggers = fx.sample_trigger_set(n_activators=1)
constraints = fx.sample_constraints()

for t in triggers.activators:
    print(f'{t.trigger_id}  {t.symbol:6}  {t.sequence}  {t.length} nt')
print('arity:', triggers.arity, '| logic:', triggers.logic_type)

## `required_tools()` — implemented

In [ ]:
fx.attempt('required_tools', gate.required_tools)

## `is_compatible()` — host is checked first

"CRISPR is not available in E. coli" tells the researcher to change organism;
"not implemented yet" tells them to wait. Compare the two hosts.

*(While `available = False`, `supports()` is `False` for every host, so both
currently report the host message. They separate once the family is switched on.)*

In [ ]:
fx.attempt('is_compatible (human)', lambda: gate.is_compatible(triggers, constraints))
ecoli_gate = fx.crispr(host=fx.Host.ECOLI)
fx.attempt('is_compatible (ecoli)', lambda: ecoli_gate.is_compatible(triggers, constraints))

## `generate_designs()` *(Step 5)*

In [ ]:
designs = fx.attempt(
    'generate_designs',
    lambda: list(gate.generate_designs(triggers, constraints)),
)

## `evaluate_design()` *(Step 5)*

In [ ]:
target = designs[0] if designs else fx.sample_design(gate, triggers)
fx.attempt('evaluate_design', lambda: gate.evaluate_design(target))

## `emit_sequence()` and `describe()` — output helpers

These two are implemented today. `generate_designs()` is not, so
`fx.sample_design(...)` hands us a plausible `GateDesign` to call them on.

In [ ]:
design = fx.sample_design(gate, triggers)
print('design_id  ', design.design_id)
print('gate_kind  ', design.gate_kind)
print('length     ', design.length, 'nt')
print('emit_sequence:', gate.emit_sequence(design))
print('describe     :', gate.describe(design))

## Switching in real folding

Everything above runs against `fx.StubFoldEngine` — deterministic, fake, no
ViennaRNA. For genuine structure predictions, pass `real_fold=True` when you build
the gate (needs `import RNA` to work in this environment):

```python
gate = fx.crispr(host=host, real_fold=True)
folder = fx.fold_engine(real=True)
folder.mfe('GGGAAACCCUUUGGGAAACCC')   # -> FoldResult(structure, energy)
```